In [ ]:
import cv2 as cv
import numpy as np

def zoom_nearest(img, s):
    """Zoom image using nearest-neighbor interpolation."""
    h, w = img.shape[:2]
    new_h, new_w = int(h * s), int(w * s)
    zoomed = np.zeros((new_h, new_w), dtype=img.dtype)
    
    for i in range(new_h):
        for j in range(new_w):
            # Map back to original coordinates
            src_i = int(i / s)
            src_j = int(j / s)
            zoomed[i, j] = img[src_i, src_j]
    return zoomed

def zoom_bilinear(img, s):
    """Zoom image using bilinear interpolation."""
    h, w = img.shape[:2]
    new_h, new_w = int(h * s), int(w * s)
    zoomed = np.zeros((new_h, new_w), dtype=np.float32)
    
    for i in range(new_h):
        for j in range(new_w):
            # Map back to original coordinates (float)
            src_i = i / s
            src_j = j / s
            
            i0, j0 = int(np.floor(src_i)), int(np.floor(src_j))
            i1, j1 = min(i0+1, h-1), min(j0+1, w-1)
            
            di, dj = src_i - i0, src_j - j0
            
            # Bilinear interpolation
            top = (1-dj)*img[i0, j0] + dj*img[i0, j1]
            bottom = (1-dj)*img[i1, j0] + dj*img[i1, j1]
            zoomed[i, j] = (1-di)*top + di*bottom
    
    return np.clip(zoomed, 0, 255).astype(img.dtype)

def normalized_ssd(img1, img2):
    """Compute normalized sum of squared differences between two images."""
    diff = (img1.astype(np.float32) - img2.astype(np.float32))**2
    return np.sum(diff) / np.sum(img1.astype(np.float32)**2)

# Example usage

small = cv.imread('/Users/sahansach/Documents/MSC/Semester 3/Computer Vision/Assignment 01/a1images/a1q8images/im01small.png', cv.IMREAD_REDUCED_GRAYSCALE_2)
assert small is not None


large = cv.imread('/Users/sahansach/Documents/MSC/Semester 3/Computer Vision/Assignment 01/a1images/a1q8images/im01.png', cv.IMREAD_REDUCED_GRAYSCALE_2)
assert large is not None

s = large.shape[0] / small.shape[0]  # zoom factor based on height

zoom_nn = zoom_nearest(small, s)
zoom_bi = zoom_bilinear(small, s)

ssd_nn = normalized_ssd(large, zoom_nn)
ssd_bi = normalized_ssd(large, zoom_bi)

print("Normalized SSD (Nearest Neighbor):", ssd_nn)
print("Normalized SSD (Bilinear):", ssd_bi)




Normalized SSD (Nearest Neighbor): 0.019357825
Normalized SSD (Bilinear): 0.026483465


Both are small relative to the denominator, confirming that the zoomed image is a good approximation of the original.